# 🌳 RadixAttention 深度解析 — Token-level 前缀缓存

**本文目标**：深入理解 RadixAttention 的 Radix Tree 数据结构、KV Cache 分配策略和与 PagedAttention 的精确对比。

读完这篇你会理解：
- Radix Tree 的节点结构和操作算法
- Token-level 内存分配的优劣
- 引用计数和 LRU 淘汰的协作机制
- 与 vLLM PagedAttention 的定量对比

## 1. Radix Tree 数据结构

### 1.1 核心定义

```python
class TreeNode:
    """Radix Tree 节点 = 一段连续的 token 序列 + KV Cache"""
    
    # token 序列
    token_ids: List[int]         # 该节点存储的 tokens
    children: Dict[int, TreeNode] # 子节点 (key=下一个 token ID)
    parent: Optional[TreeNode]
    
    # KV Cache
    kv_cache: Optional[KVCache]  # 本节点的 KV Cache (可变大小!)
    
    # 管理
    ref_count: int               # 被多少 Sequence 引用
    last_access_time: float      # LRU 淘汰用
    
    # 匹配
    def match(self, tokens: List[int]) -> int:
        """返回前缀匹配长度 (0 ~ len(self.token_ids))"""
        match_len = 0
        for t1, t2 in zip(self.token_ids, tokens):
            if t1 != t2:
                break
            match_len += 1
        return match_len
```

### 1.2 Radix Tree 的关键操作

```python
class RadixCache:
    def __init__(self):
        self.root = TreeNode(token_ids=[])  # 空根节点
    
    def insert(self, tokens: List[int], kv_cache_data):
        """插入一个新的 token 序列 + KV Cache"""
        node = self.root
        offset = 0
        
        while offset < len(tokens):
            # 1. 在子节点中找匹配
            next_token = tokens[offset]
            
            if next_token in node.children:
                child = node.children[next_token]
                match_len = child.match(tokens[offset:])
                
                if match_len == len(child.token_ids):
                    # 完全匹配 → 继续往下走
                    child.ref_count += 1
                    offset += match_len
                    node = child
                else:
                    # 部分匹配 → 需要分裂节点!
                    self._split_node(child, match_len)
                    # 分裂后继续插入
            else:
                # 2. 没有匹配 → 创建新节点
                remaining = tokens[offset:]
                new_node = TreeNode(
                    token_ids=remaining,
                    kv_cache=kv_cache_data[offset:],
                    ref_count=1
                )
                node.children[remaining[0]] = new_node
                break
    
    def _split_node(self, node: TreeNode, split_pos: int):
        """在 split_pos 处分裂节点 (部分匹配处理)"""
        # 原节点: [A, B, C, D, E]
        # 分裂为: [A, B] → [C, D, E]
        prefix_tokens = node.token_ids[:split_pos]
        suffix_tokens = node.token_ids[split_pos:]
        
        # 创建前缀节点 (共享部分)
        prefix_node = TreeNode(
            token_ids=prefix_tokens,
            kv_cache=node.kv_cache[:split_pos],
            ref_count=node.ref_count
        )
        
        # 原节点变成后缀
        node.token_ids = suffix_tokens
        node.kv_cache = node.kv_cache[split_pos:]
        
        # 重建父子关系
        # parent → prefix_node → node (suffix)
        prefix_node.parent = node.parent
        prefix_node.children[suffix_tokens[0]] = node
        node.parent = prefix_node
```

### 1.3 为什么 Radix Tree 适合前缀共享？

```
PagedAttention (block table):
  请求 A: [Block 42] → [Block 17] → [Block 3]
  请求 B: [Block 42] → [Block 17] → [Block 99]  ← 新 block
  → 注意: Block 42 和 17 被两个请求共享 (CoW)
  → 但如果 Block 17 只被部分共享? → 无法处理!

  block table 是一个 "线性映射表"
  → 只能做 "整 block 共享"
  → 灵活性 = block_size 粒度

RadixAttention (radix tree):
  请求 A: Root → "You" → " are" → " helpful." → ...
  请求 B: Root → "You" → " are" → " helpful" → ...
  → "You are" 自动共享 (3 tokens)
  → "helpful" 部分匹配 (8 个 token 中的 7 个相同)
    → 分裂节点: "help" (共享) + "ful." (A) vs. "ful" (B)

  radix tree 是一个 "压缩前缀树"
  → 任何长度的公共前缀都被自动发现和共享
  → 灵活性 = token 级别粒度
```

## 2. KV Cache 管理与引用计数

### 2.1 Radix Tree 节点的生命周期

```
节点的创建: insert() 时创建 → ref_count=1
节点的共享: match() 命中 → ref_count += 1
节点的释放: Sequence 完成 → ref_count -= 1
节点的回收: ref_count == 0 + LRU 淘汰 → 释放 KV Cache

具体流程:

1. 请求 A 到达: "You are helpful"
   → Root → "You" → " are" → " helpful"
   → 创建 3 个节点, 每个 ref_count=1

2. 请求 B 到达: "You are great"
   → Root → "You" → " are" → 匹配! (ref_count += 1)
   → " great" (新节点, ref_count=1)
   → "You" ref_count=2, " are" ref_count=2

3. 请求 A 完成:
   → 沿路径递减 ref_count
   → "You" ref_count=2 → 1
   → " are" ref_count=2 → 1
   → " helpful" ref_count=1 → 0 → 可被 LRU 淘汰

4. 显存压力:
   → LRU: 淘汰 " helpful" (ref_count=0, 最久未访问)
   → 如果请求 A 的后续没有被其他请求引用 → 回收
```

### 2.2 引用计数的实现

```python
class RadixCache:
    def lock_node(self, node, seq_id):
        """Sequence 引用此节点 → ref_count++"""
        node.ref_count += 1
        node.locked_by.add(seq_id)
    
    def unlock_node(self, node, seq_id):
        """Sequence 释放 → ref_count--"""
        node.ref_count -= 1
        node.locked_by.discard(seq_id)
        
        if node.ref_count == 0:
            # 无引用 → 加入 LRU 淘汰候选
            self.lru_queue.append(node)
    
    def evict_lru(self, target_bytes):
        """LRU 淘汰, 释放 target_bytes 的 KV Cache"""
        freed = 0
        while freed < target_bytes and self.lru_queue:
            node = self.lru_queue.popleft()
            if node.ref_count == 0:
                freed += node.kv_cache.size_bytes
                node.kv_cache = None  # 释放 KV Cache
        return freed
```

In [1]:
# Radix Tree 前缀缓存模拟

class SimpleRadixTree:
    def __init__(self):
        self.root = {}
        self.stored_tokens = 0
        self.shared_tokens = 0
        self.total_tokens = 0

    def insert(self, tokens):
        self.total_tokens += len(tokens)
        node = self.root
        pos = 0
        while pos < len(tokens):
            t = tokens[pos]
            if t in node:
                self.shared_tokens += 1
            else:
                node[t] = {}
                self.stored_tokens += 1
            node = node[t]
            pos += 1

    @property
    def hit_rate(self):
        return self.shared_tokens / max(self.total_tokens, 1)

# 模拟: 50 个 Agent 请求
import random
random.seed(42)

tree = SimpleRadixTree()
for i in range(50):
    tokens = list(range(2000))  # system prompt
    for r in range(5):
        tokens.append(10000 + random.randint(0, 99))
        tokens.extend([30000] * 30)
        tokens.extend([40000 + random.randint(0, 9999) for _ in range(500)])
        tokens.extend([50000 + random.randint(0, 99) for _ in range(50)])
    tree.insert(tokens)

print("Radix Tree 缓存模拟 (50 Agent 请求)")
print(f"总 tokens: {tree.total_tokens:,}")
print(f"独立存储: {tree.stored_tokens:,}")
print(f"共享命中: {tree.shared_tokens:,}")
print(f"命中率: {tree.hit_rate*100:.1f}%")
print(f"节省: {(1 - tree.stored_tokens/tree.total_tokens)*100:.1f}%")
print()
print("观察: token-level 粒度使得 system prompt 的前")
print("2000 个 token 100% 命中, tool_call 固定格式也部分命中")

Radix Tree 缓存模拟 (50 Agent 请求)
总 tokens: 245,250
独立存储: 146,971
共享命中: 98,279
命中率: 40.1%
节省: 40.1%

观察: token-level 粒度使得 system prompt 的前
2000 个 token 100% 命中, tool_call 固定格式也部分命中
